# Module Fine-Tuning Évolution ADL

**Backbones :** CodeT5-base (220M) et CodeT5plus-770M issus du fine-tuning de compréhension  
**Dataset :** Evolution Dataset V4 (1804 paires — 6 opérations — ACME + AADL)  
**Stratégies :** A = Full Fine-Tuning | B = Encodeur Gelé  

| # | Modèle | Stratégie | Paramètres entraînables |
|---|--------|-----------|------------------------|
| 1 | CodeT5-base | A — Full FT | 220M (100%) |
| 2 | CodeT5-base | B — Encodeur Gelé | ~110M (50%) |
| 3 | CodeT5plus-770M | A — Full FT | 770M (100%) |
| 4 | CodeT5plus-770M | B — Encodeur Gelé | ~385M (50%) |

**Score composite évolution (sélection finale) :**
```
score = 0.30×BERT-F1 + 0.25×METEOR + 0.20×ROUGE-L + 0.15×Validité_syntaxique + 0.10×Précision_opérationnelle
```

In [ ]:
# ============================================================
# 1 — Installation & Imports
# ============================================================
!pip install rouge_score bert_score nltk transformers \
             datasets accelerate sentencepiece -q

# Optional: Mount Google Drive if running in Colab
# from google.colab import drive
# drive.mount('/content/drive')

import os, gc, json, re, glob, torch, numpy as np
from collections import Counter

# Set Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Device selected: {device}")
if device == "cuda":
    print(f"   GPU Name : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ============================================================
# 2 — NLTK Setup
# ============================================================
import nltk
for pkg in ['wordnet', 'omw-1.4', 'punkt', 'punkt_tab']:
    nltk.download(pkg, quiet=True)
print("[INFO] NLTK data downloaded and ready.")

In [ ]:
# ============================================================
# 3 — Centralized Paths
# ============================================================
# Update this path if running on Colab (e.g., "/content/drive/MyDrive/ArchPipeline")
BASE_DIR = "." 

# ── Evolution Dataset V4 ─────────────────────────────────────
EVO_DIR   = f"{BASE_DIR}/data/evolution"
EVO_TRAIN = f"{EVO_DIR}/evo_train.jsonl"
EVO_VAL   = f"{EVO_DIR}/evo_val.jsonl"
EVO_TEST  = f"{EVO_DIR}/evo_test_v2.jsonl"

# ── Backbones from Comprehension Phase ──────────────────────
# CodeT5-base  : Best comprehension model (EM=0.3250, BLEU=0.5496)
# CodeT5+ 770M : 2nd best (ROUGE-L=0.6851, BERT-F1=0.9519)
MODEL_CT5B_COMPR  = f"{BASE_DIR}/models/finetuned_CodeT5base" 
MODEL_CT5P_COMPR  = f"{BASE_DIR}/models/finetuned_CodeT5plus_770m" 

# ── Output Directories for 4 configurations ─────────────────
OUT_CT5B_A = f"{BASE_DIR}/results/evo_CodeT5base_fullFT" 
OUT_CT5B_B = f"{BASE_DIR}/results/evo_CodeT5base_frozenEnc" 
OUT_CT5P_A = f"{BASE_DIR}/results/evo_CodeT5plus_fullFT" 
OUT_CT5P_B = f"{BASE_DIR}/results/evo_CodeT5plus_frozenEnc" 

# Create directories if they don't exist
for d in [OUT_CT5B_A, OUT_CT5B_B, OUT_CT5P_A, OUT_CT5P_B]:
    os.makedirs(d, exist_ok=True)

# Verify file existence
print("[INFO] Verifying file paths:")
files_to_check = [
    ("evo_train.jsonl", EVO_TRAIN),
    ("evo_val.jsonl", EVO_VAL),
    ("evo_test.jsonl", EVO_TEST),
    ("CodeT5-base comprehension", MODEL_CT5B_COMPR),
    ("CodeT5+ 770M comprehension", MODEL_CT5P_COMPR),
]
for label, path in files_to_check:
    status = "[OK]" if os.path.exists(path) else "[MISSING]"
    print(f"  {status} {label:<35} -> {path}")

In [ ]:
# ============================================================
# 4 — JSONL Loading & Preprocessing
# ============================================================
from datasets import Dataset
from transformers import (
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, EarlyStoppingCallback,
)

def load_jsonl(path: str) -> list:
    """Load data from a JSONL file."""
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def load_evolution_datasets():
    """Load train, val, and test sets, and print test distribution."""
    train = load_jsonl(EVO_TRAIN)
    val   = load_jsonl(EVO_VAL)
    test  = load_jsonl(EVO_TEST)
    
    print(f"[INFO] Train size: {len(train)} | Val size: {len(val)} | Test size: {len(test)}")
    ops = Counter(ex.get("operation", "?") for ex in test)
    print("[INFO] Test distribution by operation:")
    for op, n in sorted(ops.items()):
        print(f"     {op:<25} : {n}")
        
    return train, val, test

def build_evolution_preprocessor(tokenizer, max_input: int = 256, max_target: int = 256):
    """
    Preprocessing function for the evolution task.
    Input  : "evolve: [OP] {op} [ADL] {adl_before} [REQ] {request}"
    Target : Evolved ADL
    """
    def preprocess(examples):
        model_inputs = tokenizer(
            examples["input"],
            max_length=max_input,
            truncation=True,
            padding=False,
        )
        labels = tokenizer(
            text_target=examples["target"],
            max_length=max_target,
            truncation=True,
            padding=False,
        )
        model_inputs["labels"] = [
            [(t if t != tokenizer.pad_token_id else -100) for t in lab]
            for lab in labels["input_ids"]
        ]
        return model_inputs
    return preprocess

print("[INFO] Loading and preprocessing functions defined.")

In [ ]:
# ============================================================
# 5 — Generic TrainingArguments
# ============================================================

def get_ct5base_args(output_dir: str, num_epochs: int = 20, lr: float = 5e-5) -> Seq2SeqTrainingArguments:
    """Arguments for CodeT5-base (220M) - fp16, batch=4, accum=2."""
    return Seq2SeqTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=2,      # Effective batch size = 8
        learning_rate=lr,
        warmup_steps=100,
        weight_decay=0.01,
        optim="adamw_torch",
        lr_scheduler_type="cosine",
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        predict_with_generate=True,
        generation_max_length=256,
        fp16=torch.cuda.is_available(),
        bf16=False,
        report_to="none",
        seed=42,                            # Reproducibility seed
        dataloader_pin_memory=False,
        dataloader_num_workers=0,
    )

def get_ct5plus_args(output_dir: str, num_epochs: int = 20, lr: float = 3e-5) -> Seq2SeqTrainingArguments:
    """Arguments for CodeT5+ 770M - fp16=False (unstable), adafactor, batch=2, accum=4."""
    return Seq2SeqTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,      # Effective batch size = 8
        learning_rate=lr,
        warmup_steps=100,
        weight_decay=0.01,
        optim="adafactor",
        lr_scheduler_type="constant",
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        predict_with_generate=True,
        generation_max_length=256,
        fp16=False,                         # Unstable for 770M
        bf16=False,
        report_to="none",
        seed=42,                            # Reproducibility seed
        dataloader_pin_memory=False,
        dataloader_num_workers=0,
    )

def launch_evolution_training(model, tokenizer, train_tok, val_tok, args) -> Seq2SeqTrainer:
    """Initializes Trainer and starts training with Early Stopping."""
    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        data_collator=DataCollatorForSeq2Seq(
            tokenizer, model=model,
            padding=True, label_pad_token_id=-100),
        processing_class=tokenizer,
        callbacks=[EarlyStoppingCallback(
            early_stopping_patience=3,
            early_stopping_threshold=0.001)],
    )
    trainer.train()
    print(f"\n[INFO] Stopped at epoch: {trainer.state.epoch:.0f}/20")
    print(f"[INFO] Best validation loss: {trainer.state.best_metric:.4f}")
    return trainer

print("[INFO] TrainingArguments and launcher defined.")

In [ ]:
# ============================================================
# 6 — NLP Metrics & Manual BERTScore
# Calculated via AutoTokenizer/AutoModel roberta-large to avoid 
# tokenizer conflicts with T5/CodeT5.
# ============================================================
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer as rs_module
from nltk.translate.meteor_score import meteor_score

def compute_manual_bertscore(predictions: list, references: list, batch_size: int = 32) -> tuple:
    """
    Manual BERTScore calculation using roberta-large.
    Uses local variables to prevent tokenizer collisions with CodeT5.
    """
    from transformers import AutoTokenizer as _AT
    from transformers import AutoModel as _AM

    bs_device = "cuda" if torch.cuda.is_available() else "cpu"
    print("   Loading roberta-large for BERTScore...")
    _tok = _AT.from_pretrained("roberta-large")
    _mod = _AM.from_pretrained("roberta-large").to(bs_device)
    _mod.eval()

    def _encode(texts: list) -> torch.Tensor:
        all_embs = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i: i + batch_size]
            enc = _tok(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=256,
            ).to(bs_device)
            with torch.no_grad():
                out = _mod(**enc)
            # Mean pooling over non-padded tokens
            mask = enc["attention_mask"].unsqueeze(-1).float()
            emb = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
            all_embs.append(emb.cpu())
        return torch.cat(all_embs, dim=0)

    emb_p = _encode(predictions)
    emb_r = _encode(references)

    # L2 Normalization
    emb_p = emb_p / (emb_p.norm(dim=1, keepdim=True) + 1e-9)
    emb_r = emb_r / (emb_r.norm(dim=1, keepdim=True) + 1e-9)

    # Pairwise Cosine Similarity
    cos = (emb_p * emb_r).sum(dim=1).numpy()
    cos = np.clip(cos, -1.0, 1.0)

    # Memory cleanup
    del _mod, _tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    P = float(np.mean(cos))
    R = float(np.mean(cos))  # Symmetric (mean pooling)
    F1 = float(np.mean(cos))
    return P, R, F1

print("[INFO] Manual BERTScore function defined.")

In [ ]:
# ============================================================
# 7 — ADL Syntactic Validity
# ============================================================
ADL_KEYWORDS = {
    "component", "connector", "system", "style", "family",
    "port", "role", "property", "attachment", "binding",
    "subcomponent", "thread", "process", "porttype",
    "datatype", "eventlist", "implementation",
}

def is_syntactically_valid(adl: str) -> bool:
    """Checks if the generated ADL is syntactically well-formed."""
    if not adl or len(adl.strip()) < 5:
        return False
    if adl.count("{") != adl.count("}"):
        return False
    if re.search(r'\.{3}|…', adl):  # Check for truncation artifacts
        return False
    return any(kw in adl.lower() for kw in ADL_KEYWORDS)

def compute_syntactic_validity(predictions: list, test_data: list) -> dict:
    """Computes global, ACME, and AADL syntactic validity."""
    valid = [is_syntactically_valid(p) for p in predictions]
    acme, aadl = [], []
    
    for i, ex in enumerate(test_data):
        t = ex.get("adl_type", "").upper()
        if t == "ACME": acme.append(valid[i])
        elif t == "AADL": aadl.append(valid[i])
        
    return {
        "global_validity": float(np.mean(valid)),
        "ACME_validity": float(np.mean(acme)) if acme else 0.0,
        "AADL_validity": float(np.mean(aadl)) if aadl else 0.0,
        "valid_count": int(sum(valid)),
        "total_count": len(predictions),
    }

# ============================================================
# 8 — Operational Precision by Type
# ============================================================
def clean_field(value) -> str:
    if value is None:
        return ""
    s = str(value).strip()
    return "" if s.lower() in ("none", "nan", "null", "") else s

def verify_operation(pred: str, example: dict) -> bool:
    """Verifies if the predicted ADL correctly applies the requested operation."""
    op = example.get("operation", "").upper()
    target = clean_field(example.get("cible", ""))
    pred_l = pred.lower().strip()
    input_l = str(example.get("input", "")).lower()

    if op == "ADD_PORT":
        ports_before = len(re.findall(r'port\s+\w+', input_l))
        ports_after = len(re.findall(r'port\s+\w+', pred_l))
        if ports_after > ports_before:
            return True
        return bool(re.search(r'port\s+\w+', pred_l))

    elif op == "MODIFY_PROPERTY":
        props_before = set(re.findall(r'[\w:]+\s*[=>]+\s*[\w".]+', input_l))
        props_pred = set(re.findall(r'[\w:]+\s*[=>]+\s*[\w".]+', pred_l))
        return len(props_pred - props_before) > 0

    elif op == "ADD_COMPONENT":
        comps_before = len(re.findall(r'component\s+\w+', input_l, re.IGNORECASE))
        comps_after = len(re.findall(r'component\s+\w+', pred_l, re.IGNORECASE))
        if comps_after > comps_before:
            return True
        return bool(re.search(r'subcomponents\s+\w+', pred_l, re.IGNORECASE))

    elif op == "DELETE_COMPONENT":
        if not target:
            return len(pred_l.strip()) > 5
        return target.lower() not in pred_l

    elif op == "ADD_CONNECTOR":
        # Checks that prediction contains a connector and differs from source
        CONN_KEYWORDS = [
            "connector", "connections", "bus access", "pipe", "channel", "bus",
            "link", "bridge", "eventchannel", "asyncpipe", "msgqueue", "databus",
            "eventbus", "ctrllink", "servicebus", "netlink", "sharedbus", 
            "streampipe", "httpconn", "databridg", "servicelink", "rpcconn", 
            "asynclink", "syncbus"
        ]
        # Extract source ADL from input
        if "[adl]" in input_l:
            adl_source = input_l.split("[adl]")[-1].split("[req]")[0].strip()
        else:
            adl_source = input_l

        pred_has_conn = any(kw in pred_l for kw in CONN_KEYWORDS)
        pred_diff_source = pred_l.strip() != adl_source.strip()
        return pred_has_conn and pred_diff_source

    elif op == "DELETE_CONNECTOR":
        if not target:
            return len(pred_l.strip()) > 5
        return target.lower() not in pred_l

    return False

def compute_operational_precision(predictions: list, test_data: list) -> dict:
    """Computes operational precision per operation and globally."""
    results = {}
    correct_all, total_all = 0, 0

    for pred, ex in zip(predictions, test_data):
        op = ex.get("operation", "UNKNOWN")
        is_correct = verify_operation(pred, ex)
        if op not in results:
            results[op] = {"correct": 0, "total": 0}
        results[op]["correct"] += int(is_correct)
        results[op]["total"] += 1
        correct_all += int(is_correct)
        total_all += 1

    for op in results:
        c = results[op]["correct"]
        t = results[op]["total"]
        results[op]["precision"] = round(c / t, 4) if t > 0 else 0.0

    results["GLOBAL"] = {
        "correct": correct_all,
        "total": total_all,
        "precision": round(correct_all / total_all, 4) if total_all > 0 else 0.0,
    }
    return results

print("[INFO] Domain-specific metrics (Syntactic Validity & Operational Precision) defined.")

In [ ]:
# ============================================================
# 9 — Full Evaluation Pipeline
# ============================================================

def evaluate_evolution_model(model, tokenizer, test_data: list, model_name: str, output_dir: str) -> dict:
    """Runs generation, computes all metrics, and saves to JSON."""
    model.eval()

    # Force FP32 for generation to avoid fp16 underflows
    try:
        param_dtype = next(model.parameters()).dtype
        if param_dtype == torch.float16:
            model = model.float()
    except StopIteration:
        pass

    predictions, references = [], []

    print(f"\n[INFO] Generating predictions for {len(test_data)} examples...")
    for i, example in enumerate(test_data):
        inputs = tokenizer(
            str(example["input"]),
            return_tensors="pt",
            max_length=256,
            truncation=True,
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=256,
                num_beams=4,
                early_stopping=True,
                no_repeat_ngram_size=3,
                length_penalty=1.0,
            )
        pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
        predictions.append(pred.strip())
        references.append(str(example["target"]).strip())

        if (i + 1) % 50 == 0:
            print(f"   Generated {i+1}/{len(test_data)}...")

    print(f"[INFO] {len(predictions)} predictions completed.")

    # ── Qualitative Preview ────────────────────────────────────
    seen = set()
    print("\n--- Preview (1 example per operation) ---")
    for i, ex in enumerate(test_data):
        op = ex.get("operation", "?")
        if op not in seen:
            seen.add(op)
            print(f"  [{op}]")
            print(f"   Ref  : {references[i][:70]}")
            print(f"   Pred : {predictions[i][:70]}")

    # ── Exact Match ──────────────────────────────────────────
    em = sum(1 if p == r else 0 for p, r in zip(predictions, references)) / len(predictions)

    # ── BLEU ─────────────────────────────────────────────────
    smooth = SmoothingFunction()
    refs_bleu = [[r.split()] for r in references]
    preds_bleu = [p.split() for p in predictions]
    bleu = corpus_bleu(refs_bleu, preds_bleu, smoothing_function=smooth.method4)
    smooth_bleu = corpus_bleu(refs_bleu, preds_bleu, smoothing_function=smooth.method1)

    # ── ROUGE ────────────────────────────────────────────────
    scorer_r = rs_module.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    r1, r2, rL = [], [], []
    for p, r in zip(predictions, references):
        s = scorer_r.score(r, p)
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rL.append(s['rougeL'].fmeasure)
    rouge1, rouge2, rougeL = float(np.mean(r1)), float(np.mean(r2)), float(np.mean(rL))

    # ── METEOR ───────────────────────────────────────────────
    meteor_scores = []
    for p, r in zip(predictions, references):
        try:
            meteor_scores.append(meteor_score([r.split()], p.split()))
        except:
            meteor_scores.append(0.0)
    meteor = float(np.mean(meteor_scores))

    # ── BERTScore ────────────────────────────────────────────
    print("\n[INFO] Computing BERTScore...")
    try:
        bert_p, bert_r, bert_f1 = compute_manual_bertscore(predictions, references)
    except Exception as e:
        print(f"[WARNING] BERTScore failed: {e}")
        bert_p = bert_r = bert_f1 = 0.0

    # ── Syntactic Validity ───────────────────────────────────
    syntactic_validity = compute_syntactic_validity(predictions, test_data)

    # ── Operational Precision ────────────────────────────────
    operational_precision = compute_operational_precision(predictions, test_data)

    # ── Composite Score ──────────────────────────────────────
    global_prec = operational_precision.get("GLOBAL", {}).get("precision", 0.0)
    global_val = syntactic_validity.get("global_validity", 0.0)
    composite_score = round(
        0.30 * bert_f1 +
        0.25 * meteor +
        0.20 * rougeL +
        0.15 * global_val +
        0.10 * global_prec
    , 4)

    # ── Display Results ──────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"[RESULTS] - {model_name}")
    print(f"   Test size: {len(test_data)}")
    print(f"{'='*60}")
    for name, val in [
        ("Exact Match", em), ("BLEU", bleu), ("Smooth BLEU", smooth_bleu),
        ("ROUGE-1", rouge1), ("ROUGE-2", rouge2), ("ROUGE-L", rougeL),
        ("METEOR", meteor), ("BERT-Precision", bert_p), ("BERT-Recall", bert_r),
        ("BERT-F1", bert_f1), ("Global Validity", global_val),
        ("ACME Validity", syntactic_validity.get("ACME_validity", 0.0)),
        ("AADL Validity", syntactic_validity.get("AADL_validity", 0.0)),
        ("Global Op. Precision", global_prec), ("Composite Score", composite_score),
    ]:
        print(f"  {name:<22} : {val:.4f}")
    print(f"{'='*60}")

    print("\n[INFO] Precision by operation:")
    for op, stats in operational_precision.items():
        if op != "GLOBAL":
            print(f"  {op:<25} : {stats['precision']:.4f} ({stats['correct']}/{stats['total']})")

    # ── Save JSON ────────────────────────────────────────────
    results = {
        "model": model_name,
        "test_size": len(test_data),
        "EM": em, "BLEU": bleu, "Smooth_BLEU": smooth_bleu,
        "ROUGE-1": rouge1, "ROUGE-2": rouge2, "ROUGE-L": rougeL,
        "METEOR": meteor, "BERT-P": bert_p, "BERT-R": bert_r, "BERT-F1": bert_f1,
        "syntactic_validity": syntactic_validity,
        "operational_precision": operational_precision,
        "composite_score": composite_score,
    }
    json_path = f"{output_dir}/results_{model_name}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4, ensure_ascii=False)
    print(f"\n[INFO] Results saved to: {json_path}")

    return results

print("[INFO] Full evaluation pipeline defined.")

Model 1: CodeT5-base | Strategy A (Full Fine-Tuning)
Backbone: CodeT5-base fine-tuned on the comprehension phase.Strategy: All parameters are trainable (220M, 100%).Hyperparameters: LR = 5e-5 | Effective Batch Size = 8 | Optimizer = AdamW + cosine schedule.


In [ ]:
# ============================================================
# MODEL 1 — CodeT5-base | Strategy A: Full Fine-Tuning
# ============================================================
import gc
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Free up VRAM from previous runs
for _v in ["model_ct5b_A", "model_ct5b_B", "model_ct5p_A", "model_ct5p_B"]:
    if _v in globals():
        try:
            globals()[_v].cpu()
        except:
            pass
        del globals()[_v]
        
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    print(f"[INFO] Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

# Load Data
train_data, val_data, test_data = load_evolution_datasets()
train_ds = Dataset.from_list(train_data)
val_ds = Dataset.from_list(val_data)

# Load Comprehension Backbone
print(f"\n[INFO] Loading CodeT5-base (comprehension -> evolution A)...")
tokenizer_ct5b = AutoTokenizer.from_pretrained(MODEL_CT5B_COMPR)
model_ct5b_A = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_CT5B_COMPR, torch_dtype=torch.float32
).to(device)

total_params = sum(p.numel() for p in model_ct5b_A.parameters())
trainable_params = sum(p.numel() for p in model_ct5b_A.parameters() if p.requires_grad)
print(f"[INFO] CodeT5-base | {total_params/1e6:.0f}M total params | {trainable_params/1e6:.0f}M trainable (100%)")
if device == "cuda":
    print(f"   Allocated VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# Preprocessing
COLS_TO_REMOVE = [c for c in 
    ["input", "target", "operation", "adl_type", "difficulte", "source", "cible"] 
    if c in train_ds.column_names]

preprocess_ct5b = build_evolution_preprocessor(tokenizer_ct5b)
train_tok = train_ds.map(preprocess_ct5b, batched=True, remove_columns=COLS_TO_REMOVE)
val_tok = val_ds.map(preprocess_ct5b, batched=True, remove_columns=COLS_TO_REMOVE)

# Verification
ex = train_tok[0]
valid_tokens = sum(1 for t in ex["labels"] if t != -100)
print(f"\n[INFO] Labels check: {len(ex['labels'])} tokens, {valid_tokens} valid")
if valid_tokens == 0:
    raise ValueError("Empty labels detected - check load_evolution_datasets()")
print("[INFO] Preprocessing successful.")

# Training Execution
args_ct5b_A = get_ct5base_args(OUT_CT5B_A)

print(f"\n[INFO] Starting CodeT5-base - Strategy A (Full FT)...")
print(f"   lr=5e-5 | batch=4 | accum=2 | fp16={torch.cuda.is_available()}")

trainer_ct5b_A = launch_evolution_training(
    model_ct5b_A, tokenizer_ct5b, train_tok, val_tok, args_ct5b_A
)

# Save Model
trainer_ct5b_A.save_model(OUT_CT5B_A)
tokenizer_ct5b.save_pretrained(OUT_CT5B_A)
print(f"[INFO] Model saved to: {OUT_CT5B_A}")

In [ ]:
# ── Evaluate Model 1 ───────────────────────────────────────
# Ensure test_data is loaded
if 'test_data' not in locals():
    _, _, test_data = load_evolution_datasets()

results_ct5b_A = evaluate_evolution_model(
    model=model_ct5b_A, 
    tokenizer=tokenizer_ct5b, 
    test_data=test_data,
    model_name="CodeT5base_StratA_FullFT",
    output_dir=OUT_CT5B_A
)

Model 2: CodeT5-base | Strategy B (Frozen Encoder)
Backbone: CodeT5-base fine-tuned on the comprehension phase.Strategy: Encoder is frozen — only the decoder is trained (~110M, 50%).Justification: The encoder has already learned ADL syntax during comprehension. We leverage this and specialize only the generation decoder.Hyperparameters: LR = 8e-5 (slightly higher since fewer parameters are updated) | Effective Batch Size = 8 | Optimizer = AdamW + cosine.

In [ ]:
# ============================================================
# MODEL 2 — CodeT5-base | Strategy B: Frozen Encoder
# ============================================================
import gc
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Free up VRAM from previous model
for _v in ["model_ct5b_A", "trainer_ct5b_A"]:
    if _v in globals():
        try:
            globals()[_v].cpu()
        except:
            pass
        del globals()[_v]
        
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    print(f"[INFO] Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

# Load Comprehension Backbone
print(f"\n[INFO] Loading CodeT5-base (comprehension -> evolution B)...")
tokenizer_ct5b = AutoTokenizer.from_pretrained(MODEL_CT5B_COMPR)
model_ct5b_B = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_CT5B_COMPR, torch_dtype=torch.float32
).to(device)

# Freeze the Encoder
for param in model_ct5b_B.encoder.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in model_ct5b_B.parameters())
trainable_params = sum(p.numel() for p in model_ct5b_B.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params
print(f"[INFO] CodeT5-base | Frozen={frozen_params/1e6:.0f}M ({frozen_params/total_params*100:.0f}%) | Trainable={trainable_params/1e6:.0f}M ({trainable_params/total_params*100:.0f}%)")

# Preprocessing (reuse train_ds/val_ds if already defined)
if "train_ds" not in globals():
    train_data, val_data, test_data = load_evolution_datasets()
    train_ds = Dataset.from_list(train_data)
    val_ds = Dataset.from_list(val_data)

COLS_TO_REMOVE = [c for c in 
    ["input", "target", "operation", "adl_type", "difficulte", "source", "cible"] 
    if c in train_ds.column_names]

preprocess_ct5b = build_evolution_preprocessor(tokenizer_ct5b)
train_tok = train_ds.map(preprocess_ct5b, batched=True, remove_columns=COLS_TO_REMOVE)
val_tok = val_ds.map(preprocess_ct5b, batched=True, remove_columns=COLS_TO_REMOVE)

# Training Execution - Slightly higher LR for decoder-only training
args_ct5b_B = get_ct5base_args(OUT_CT5B_B, lr=8e-5)

print(f"\n[INFO] Starting CodeT5-base - Strategy B (Frozen Encoder)...")
print(f"   lr=8e-5 | batch=4 | accum=2 | encoder frozen")

trainer_ct5b_B = launch_evolution_training(
    model_ct5b_B, tokenizer_ct5b, train_tok, val_tok, args_ct5b_B
)

# Save Model
trainer_ct5b_B.save_model(OUT_CT5B_B)
tokenizer_ct5b.save_pretrained(OUT_CT5B_B)
print(f"[INFO] Model saved to: {OUT_CT5B_B}")

In [ ]:
# ── Evaluate Model 2 ───────────────────────────────────────
# Ensure test_data is loaded
if 'test_data' not in locals():
    _, _, test_data = load_evolution_datasets()

results_ct5b_B = evaluate_evolution_model(
    model=model_ct5b_B, 
    tokenizer=tokenizer_ct5b, 
    test_data=test_data,
    model_name="CodeT5base_StratB_FrozenEnc",
    output_dir=OUT_CT5B_B
)

Model 3: CodeT5+ 770M | Strategy A (Full Fine-Tuning)
Backbone: CodeT5+ 770M fine-tuned on the comprehension phase.Strategy: All parameters are trainable (770M, 100%).Specifics: To fit the 770M model in a 15GB VRAM GPU, we use fp16=False, Adafactor with a constant LR, gradient checkpointing, and a smaller batch size (2) with gradient accumulation (4).

In [ ]:
# ============================================================
# MODEL 3 — CodeT5+ 770M | Strategy A: Full Fine-Tuning
# ============================================================
import gc
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Free up VRAM from previous models
for _v in ["model_ct5b_B", "model_ct5b_A", "trainer_ct5b_B"]:
    if _v in globals():
        try:
            globals()[_v].cpu()
        except:
            pass
        del globals()[_v]
        
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    vram = torch.cuda.mem_get_info()[0]/1e9
    print(f"[INFO] Free VRAM: {vram:.1f} GB")
    if vram < 4.0:
        raise RuntimeError("Insufficient VRAM for 770M model. Please reset runtime.")

# Load Comprehension Backbone
print(f"\n[INFO] Loading CodeT5+ 770M (comprehension -> evolution A)...")
tokenizer_ct5p = AutoTokenizer.from_pretrained(MODEL_CT5P_COMPR, use_fast=False)
model_ct5p_A = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_CT5P_COMPR, torch_dtype=torch.float32
).to(device)

# Enable Gradient Checkpointing to save ~40% VRAM
model_ct5p_A.gradient_checkpointing_enable()
model_ct5p_A.config.use_cache = False # Required for gradient checkpointing

total_params = sum(p.numel() for p in model_ct5p_A.parameters())
trainable_params = sum(p.numel() for p in model_ct5p_A.parameters() if p.requires_grad)
print(f"[INFO] CodeT5+ 770M | {total_params/1e6:.0f}M total params | {trainable_params/1e6:.0f}M trainable (100%)")
if device == "cuda":
    print(f"   Allocated VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# Preprocessing
if "train_ds" not in globals():
    train_data, val_data, test_data = load_evolution_datasets()
    train_ds = Dataset.from_list(train_data)
    val_ds = Dataset.from_list(val_data)

COLS_TO_REMOVE = [c for c in 
    ["input", "target", "operation", "adl_type", "difficulte", "source", "cible"] 
    if c in train_ds.column_names]

preprocess_ct5p = build_evolution_preprocessor(tokenizer_ct5p)
train_tok_ct5p = train_ds.map(preprocess_ct5p, batched=True, remove_columns=COLS_TO_REMOVE)
val_tok_ct5p = val_ds.map(preprocess_ct5p, batched=True, remove_columns=COLS_TO_REMOVE)

# Training Execution
args_ct5p_A = get_ct5plus_args(OUT_CT5P_A)

print(f"\n[INFO] Starting CodeT5+ 770M - Strategy A (Full FT)...")
print(f"   fp16=False | adafactor | constant LR | grad_ckpt | batch=2+accum=4")

trainer_ct5p_A = launch_evolution_training(
    model_ct5p_A, tokenizer_ct5p, train_tok_ct5p, val_tok_ct5p, args_ct5p_A
)

# Re-enable cache for inference and save
model_ct5p_A.config.use_cache = True
trainer_ct5p_A.save_model(OUT_CT5P_A)
tokenizer_ct5p.save_pretrained(OUT_CT5P_A)
print(f"[INFO] Model saved to: {OUT_CT5P_A}")

In [ ]:
# ── Evaluate Model 3 ───────────────────────────────────────
# Ensure test_data is loaded
if 'test_data' not in locals():
    _, _, test_data = load_evolution_datasets()

# Ensure cache is enabled for generation
model_ct5p_A.config.use_cache = True

results_ct5p_A = evaluate_evolution_model(
    model=model_ct5p_A, 
    tokenizer=tokenizer_ct5p, 
    test_data=test_data,
    model_name="CodeT5plus_StratA_FullFT",
    output_dir=OUT_CT5P_A
)

Model 4: CodeT5+ 770M | Strategy B (Frozen Encoder)
Backbone: CodeT5+ 770M fine-tuned on the comprehension phase.Strategy: Encoder is frozen — only the decoder is trained (~385M, 50%).Advantage: Faster training and less risk of overfitting on the 1,754 training pairs.Specifics: We set save_total_limit=1 to prevent disk space issues with large 770M checkpoints.

In [ ]:
# ============================================================
# MODEL 4 — CodeT5+ 770M | Strategy B: Frozen Encoder
# ============================================================
import gc
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Free up VRAM from previous models
for _v in ["model_ct5p_A", "trainer_ct5p_A", "model_ct5p_B", "trainer_ct5p_B",
           "train_tok_ct5p", "val_tok_ct5p"]:
    if _v in globals():
        try:
            globals()[_v].cpu()
        except:
            pass
        del globals()[_v]
        
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    vram = torch.cuda.mem_get_info()[0]/1e9
    print(f"[INFO] Free VRAM: {vram:.1f} GB")
    if vram < 4.0:
        raise RuntimeError("Insufficient VRAM for 770M model. Please reset runtime.")

# Load Comprehension Backbone
print(f"\n[INFO] Loading CodeT5+ 770M...")
tokenizer_ct5p = AutoTokenizer.from_pretrained(MODEL_CT5P_COMPR, use_fast=False)
model_ct5p_B = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_CT5P_COMPR, torch_dtype=torch.float32
).to(device)

# Freeze the Encoder
for param in model_ct5p_B.encoder.parameters():
    param.requires_grad = False

# Enable Gradient Checkpointing
model_ct5p_B.gradient_checkpointing_enable()
model_ct5p_B.config.use_cache = False

total_params = sum(p.numel() for p in model_ct5p_B.parameters())
trainable_params = sum(p.numel() for p in model_ct5p_B.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params
print(f"[INFO] Frozen={frozen_params/1e6:.0f}M ({frozen_params/total_params*100:.0f}%) | Trainable={trainable_params/1e6:.0f}M ({trainable_params/total_params*100:.0f}%)")

# Preprocessing
if "train_ds" not in globals():
    train_data, val_data, test_data = load_evolution_datasets()
    train_ds = Dataset.from_list(train_data)
    val_ds = Dataset.from_list(val_data)

COLS_TO_REMOVE = [c for c in 
    ["input", "target", "operation", "adl_type", "difficulte", "source", "cible"] 
    if c in train_ds.column_names]

if "preprocess_ct5p" not in globals():
    preprocess_ct5p = build_evolution_preprocessor(tokenizer_ct5p)

train_tok_ct5p = train_ds.map(preprocess_ct5p, batched=True, remove_columns=COLS_TO_REMOVE)
val_tok_ct5p = val_ds.map(preprocess_ct5p, batched=True, remove_columns=COLS_TO_REMOVE)

# Training Arguments
args_ct5p_B = get_ct5plus_args(OUT_CT5P_B, lr=5e-5)
args_ct5p_B.save_total_limit = 1  # Prevent disk full with large 770M checkpoints

print(f"\n[INFO] Starting CodeT5+ 770M - Strategy B (Frozen Encoder)...")
print(f"   lr=5e-5 | save_total_limit=1 | encoder frozen")

trainer_ct5p_B = launch_evolution_training(
    model_ct5p_B, tokenizer_ct5p, train_tok_ct5p, val_tok_ct5p, args_ct5p_B
)

# Save Model
model_ct5p_B.config.use_cache = True
trainer_ct5p_B.save_model(OUT_CT5P_B)
tokenizer_ct5p.save_pretrained(OUT_CT5P_B)
print(f"[INFO] Model 4 saved to: {OUT_CT5P_B}")
print(f"[INFO] Stopped at epoch: {trainer_ct5p_B.state.epoch:.0f}")
print(f"[INFO] Best loss: {trainer_ct5p_B.state.best_metric:.4f}")

In [ ]:
# ── Evaluate Model 4 ───────────────────────────────────────
# Ensure test_data is loaded
if 'test_data' not in locals():
    _, _, test_data = load_evolution_datasets()

# Ensure cache is enabled for generation
model_ct5p_B.config.use_cache = True

results_ct5p_B = evaluate_evolution_model(
    model=model_ct5p_B, 
    tokenizer=tokenizer_ct5p, 
    test_data=test_data,
    model_name="CodeT5plus_StratB_FrozenEnc",
    output_dir=OUT_CT5P_B
)

Ablation Study: Direct Evolution (CodeT5-base | Full FT)
Goal: Verify the benefit of the two-phase transfer learning strategy.Configuration B: The model is initialized directly from the general HuggingFace CodeT5-base weights, bypassing the comprehension phase entirely.Hyperparameters: Identical to Model 1 (Strategy A) to ensure a strict single-variable comparison.

Note on Tokenizer: To prevent vocabulary mismatches, we use the tokenizer from the comprehension fine-tuned model (which has identical vocab to the base model) but load the raw, untrained weights from Salesforce/codet5-base.

In [ ]:
# ============================================================
# ABLATION STUDY — Direct Evolution: CodeT5-base | Full FT
# ============================================================
import gc
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Define output path for ablation
OUT_ABLATION = f"{BASE_DIR}/results/evo_ablation_DirectEvolution_CodeT5base_FullFT"
os.makedirs(OUT_ABLATION, exist_ok=True)

# Free up VRAM from previous models
for _v in ["model_ct5b_A", "model_ct5b_B", "model_ct5p_A", "model_ct5p_B",
           "trainer_ct5b_A", "trainer_ct5b_B", "trainer_ct5p_A", "trainer_ct5p_B",
           "model_abl", "trainer_abl"]:
    if _v in globals():
        try:
            globals()[_v].cpu()
        except:
            pass
        del globals()[_v]
        
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    print(f"[INFO] Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

# Load Data
if "train_ds" not in globals():
    train_data, val_data, test_data = load_evolution_datasets()
    train_ds = Dataset.from_list(train_data)
    val_ds = Dataset.from_list(val_data)

print(f"\n{'='*70}")
print("[ABLATION STUDY] Direct Evolution")
print(f"{'='*70}")
print("Configuration: Initialization from GENERAL HuggingFace weights")
print("(NO comprehension phase transfer)")
print(f"{'='*70}")

# Load Tokenizer (from comprehension model to ensure exact vocab match)
# and Model weights (from general HuggingFace repository)
print("\n[INFO] Loading CodeT5-base (GENERAL HuggingFace weights)...")
tokenizer_abl = AutoTokenizer.from_pretrained(MODEL_CT5B_COMPR, use_fast=False)

model_abl = AutoModelForSeq2SeqLM.from_pretrained(
    "Salesforce/codet5-base", 
    torch_dtype=torch.float32
).to(device)

# Verify vocabulary compatibility
assert tokenizer_abl.vocab_size == model_abl.config.vocab_size, \
    f"Vocab mismatch: tokenizer={tokenizer_abl.vocab_size}, model={model_abl.config.vocab_size}"

total_params = sum(p.numel() for p in model_abl.parameters())
trainable_params = sum(p.numel() for p in model_abl.parameters() if p.requires_grad)
print(f"[INFO] CodeT5-base (GENERAL) | {total_params/1e6:.0f}M total params | {trainable_params/1e6:.0f}M trainable (100%)")
print(f"[INFO] Tokenizer compatible: vocab_size={tokenizer_abl.vocab_size}")
if device == "cuda":
    print(f"   Allocated VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# Preprocessing (Identical to Model 1)
COLS_TO_REMOVE = [c for c in 
    ["input", "target", "operation", "adl_type", "difficulte", "source", "cible"] 
    if c in train_ds.column_names]

preprocess_abl = build_evolution_preprocessor(tokenizer_abl)
train_tok_abl = train_ds.map(preprocess_abl, batched=True, remove_columns=COLS_TO_REMOVE)
val_tok_abl = val_ds.map(preprocess_abl, batched=True, remove_columns=COLS_TO_REMOVE)

# Verify labels
ex = train_tok_abl[0]
valid_tokens = sum(1 for t in ex["labels"] if t != -100)
print(f"\n[INFO] Labels check: {len(ex['labels'])} tokens, {valid_tokens} valid")
if valid_tokens == 0:
    raise ValueError("Empty labels detected - check load_evolution_datasets()")
print("[INFO] Preprocessing successful.")

# Training Execution (Identical to Model 1)
args_abl = get_ct5base_args(OUT_ABLATION, lr=5e-5)

print(f"\n[INFO] Starting Direct Evolution - CodeT5-base | Full FT (Ablation)...")
print(f"   lr=5e-5 | batch=4 | accum=2 | fp16={torch.cuda.is_available()}")
print(f"   [Note] Identical hyperparameters as Model 1. Only initialization differs.")

trainer_abl = launch_evolution_training(
    model_abl, tokenizer_abl, train_tok_abl, val_tok_abl, args_abl
)

# Save Model
trainer_abl.save_model(OUT_ABLATION)
tokenizer_abl.save_pretrained(OUT_ABLATION)
print(f"[INFO] Ablation model saved to: {OUT_ABLATION}")

In [ ]:
# ── Evaluate Ablation Model ─────────────────────────────────
if "test_data" not in globals():
    _, _, test_data = load_evolution_datasets()

results_abl = evaluate_evolution_model(
    model=model_abl, 
    tokenizer=tokenizer_abl, 
    test_data=test_data,
    model_name="Ablation_DirectEvolution_CodeT5base_FullFT",
    output_dir=OUT_ABLATION
)

# ── Quick Comparative Display ───────────────────────────────
print(f"\n{'='*70}")
print("[RESULTS] ABLATION COMPARISON - Key Metrics")
print(f"{'='*70}")

# Load ArchPipeline (Model 1) results if available
path_archpipeline = f"{OUT_CT5B_A}/results_CodeT5base_StratA_FullFT.json"
if os.path.exists(path_archpipeline):
    with open(path_archpipeline) as f:
        res_arch = json.load(f)

    print(f"\n{'Metric':<25} {'ArchPipeline (A)':>18} {'Direct Evo (B)':>18} {'Delta':>10}")
    print("-" * 75)

    for metric in ["BERT-F1", "METEOR", "ROUGE-L", "composite_score"]:
        val_a = res_arch.get(metric, 0.0)
        val_b = results_abl.get(metric, 0.0)
        delta = val_a - val_b
        symbol = "[+]" if delta > 0 else "[-]" if delta < 0 else "[=]"
        print(f"{metric:<25} {val_a:>18.4f} {val_b:>18.4f} {symbol}{abs(delta):>7.4f}")

    # Syntactic Validity
    val_a = res_arch.get("syntactic_validity", {}).get("global_validity", 0.0)
    val_b = results_abl.get("syntactic_validity", {}).get("global_validity", 0.0)
    delta = val_a - val_b
    symbol = "[+]" if delta > 0 else "[-]" if delta < 0 else "[=]"
    print(f"{'Syntactic Validity':<25} {val_a:>18.4f} {val_b:>18.4f} {symbol}{abs(delta):>7.4f}")

    # Operational Precision
    val_a = res_arch.get("operational_precision", {}).get("GLOBAL", {}).get("precision", 0.0)
    val_b = results_abl.get("operational_precision", {}).get("GLOBAL", {}).get("precision", 0.0)
    delta = val_a - val_b
    symbol = "[+]" if delta > 0 else "[-]" if delta < 0 else "[=]"
    print(f"{'Operational Precision':<25} {val_a:>18.4f} {val_b:>18.4f} {symbol}{abs(delta):>7.4f}")

    print(f"\n{'='*70}")
    score_a = res_arch.get("composite_score", 0.0)
    score_b = results_abl.get("composite_score", 0.0)
    if score_a > score_b:
        print(f"[CONCLUSION] ArchPipeline (transfer) outperforms Direct Evolution by {score_a - score_b:.4f}")
        print(f"   -> Comprehension transfer is BENEFICIAL")
    elif score_a < score_b:
        print(f"[WARNING] Direct Evolution outperforms ArchPipeline by {score_b - score_a:.4f}")
        print(f"   -> Comprehension transfer is HARMFUL (investigate)")
    else:
        print(f"[CONCLUSION] Equivalent scores - transfer provides no significant gain")
    print(f"{'='*70}")
else:
    print(f"[WARNING] ArchPipeline results not found at {path_archpipeline}")
    print(f"   Please run Model 1 evaluation first to generate the comparison.")

Final Comparative Results (4 Configurations)
This cell aggregates the JSON results saved during the evaluation of the four configurations and prints comparative tables for NLP metrics, Syntactic Validity, and Operational Precision. An asterisk (*) denotes the best score for each metric.

In [ ]:
# ============================================================
# FINAL COMPARATIVE TABLE — 4 configurations
# ============================================================
import glob as _glob

CONFIGS = {
    "CodeT5-base StratA (FullFT)"  : OUT_CT5B_A,
    "CodeT5-base StratB (Frozen)"  : OUT_CT5B_B,
    "CodeT5+ 770M StratA (FullFT)" : OUT_CT5P_A,
    "CodeT5+ 770M StratB (Frozen)" : OUT_CT5P_B,
}

all_results = {}
print("[INFO] Loading evaluation results...")
for name, path in CONFIGS.items():
    json_files = _glob.glob(f"{path}/results_*.json")
    if json_files:
        with open(json_files[0]) as f:
            all_results[name] = json.load(f)
        print(f"  [OK] {name}")
    else:
        print(f"  [MISSING] {name} — results not found in {path}")

if not all_results:
    raise FileNotFoundError("No results found. Please run the evaluations first.")

models = list(all_results.keys())
W = 28 # Column width for formatting

def print_table(title, rows):
    print("\n" + "="*(22 + W*len(models)))
    print(f"[RESULTS] {title}")
    print("="*(22 + W*len(models)))
    header = f"  {'Metric':<20}"
    for m in models: 
        header += f"{m[:W]:>{W}}"
    print(header)
    print("  " + "-"*20 + "-"*W*len(models))
    for label, vals in rows:
        vals_clean = [v for v in vals if v == v]
        max_v = max(vals_clean) if vals_clean else 0
        line = f"  {label:<20}"
        for v in vals:
            flag = "*" if abs(v - max_v) < 1e-6 else " "
            line += f"{flag+f'{v:.4f}':>{W}}"
        print(line)
    print("="*(22 + W*len(models)))
    print("  * = best score")

# ── NLP Metrics Table ───────────────────────────────────────
nlp_rows = []
for m in ["EM", "BLEU", "Smooth_BLEU", "ROUGE-1", "ROUGE-2",
          "ROUGE-L", "METEOR", "BERT-F1", "composite_score"]:
    nlp_rows.append((m, [all_results[n].get(m, 0.0) for n in models]))
print_table("NLP METRICS - ADL EVOLUTION", nlp_rows)

# ── Syntactic Validity Table ────────────────────────────────
val_rows = []
for k in ["global_validity", "ACME_validity", "AADL_validity"]:
    val_rows.append((k, [
        all_results[n].get("syntactic_validity", {}).get(k, 0.0)
        for n in models]))
print_table("ADL SYNTACTIC VALIDITY", val_rows)

# ── Operational Precision Table ─────────────────────────────
OPS = ["ADD_PORT", "MODIFY_PROPERTY", "ADD_COMPONENT",
       "DELETE_COMPONENT", "ADD_CONNECTOR", "DELETE_CONNECTOR", "GLOBAL"]
op_rows = []
for op in OPS:
    op_rows.append((op, [
        all_results[n].get("operational_precision", {})
                     .get(op, {}).get("precision", 0.0)
        for n in models]))
print_table("OPERATIONAL PRECISION (by operation)", op_rows)

Weight Sensitivity Analysis
Objective: Demonstrate that the ranking of the best model is invariant to the specific values of the weights. If the top model remains #1 across all configurations, the selection is not an artifact of the weighting scheme.

Six configurations are tested:

Nominal (0.30 / 0.25 / 0.20 / 0.15 / 0.10)
Uniform (0.20 / 0.20 / 0.20 / 0.20 / 0.20)
BERT-dominant (0.50 / 0.20 / 0.15 / 0.10 / 0.05)
No Validity (0.35 / 0.30 / 0.25 / 0.00 / 0.10)
Pure NLP (0.30 / 0.25 / 0.25 / 0.00 / 0.00)
Structure-first (0.20 / 0.15 / 0.15 / 0.35 / 0.15)

In [ ]:
# ============================================================
# WEIGHT SENSITIVITY ANALYSIS - Q1 ARGUMENT
# ============================================================
import numpy as np
import json
import os

# ── Check available results ──────────────────────────────────
if 'all_results' not in globals() or not all_results:
    raise RuntimeError(
        "[ERROR] Variable 'all_results' not found.\n"
        "   Please run the 'FINAL COMPARATIVE RESULTS' cell first."
    )

# ── Extract metrics for each model ───────────────────────────
def extract_metrics(res: dict) -> dict:
    return {
        "bert_f1": res.get("BERT-F1", 0.0),
        "meteor": res.get("METEOR", 0.0),
        "rouge_l": res.get("ROUGE-L", 0.0),
        "syn_val": res.get("syntactic_validity", {}).get("global_validity", 0.0),
        "op_prec": res.get("operational_precision", {}).get("GLOBAL", {}).get("precision", 0.0),
    }

metrics_dict = {name: extract_metrics(res) for name, res in all_results.items()}

# ── Define the 6 weight configurations ──────────────────────
WEIGHT_CONFIGS = {
    "Nominal        (0.30/0.25/0.20/0.15/0.10)": {
        "bert_f1": 0.30, "meteor": 0.25, "rouge_l": 0.20, "syn_val": 0.15, "op_prec": 0.10
    },
    "Uniform        (0.20/0.20/0.20/0.20/0.20)": {
        "bert_f1": 0.20, "meteor": 0.20, "rouge_l": 0.20, "syn_val": 0.20, "op_prec": 0.20
    },
    "BERT dominant  (0.50/0.20/0.15/0.10/0.05)": {
        "bert_f1": 0.50, "meteor": 0.20, "rouge_l": 0.15, "syn_val": 0.10, "op_prec": 0.05
    },
    "No validity    (0.35/0.30/0.25/0.00/0.10)": {
        "bert_f1": 0.35, "meteor": 0.30, "rouge_l": 0.25, "syn_val": 0.00, "op_prec": 0.10
    },
    "Pure NLP       (0.40/0.33/0.27/0.00/0.00)": {
        "bert_f1": 0.40, "meteor": 0.33, "rouge_l": 0.27, "syn_val": 0.00, "op_prec": 0.00
    },
    "Structure first(0.20/0.15/0.15/0.35/0.15)": {
        "bert_f1": 0.20, "meteor": 0.15, "rouge_l": 0.15, "syn_val": 0.35, "op_prec": 0.15
    },
}

# Verify weights sum to 1.0
for cfg_name, weights in WEIGHT_CONFIGS.items():
    s = sum(weights.values())
    assert abs(s - 1.0) < 1e-9, f"Invalid weights for '{cfg_name}': sum = {s:.4f}"

# ── Calculate scores for each configuration ─────────────────
def calculate_score(metr: dict, weights: dict) -> float:
    return round(sum(weights[k] * metr[k] for k in weights), 4)

sensitivity_results = {}
for cfg_name, weights in WEIGHT_CONFIGS.items():
    cfg_scores = {name: calculate_score(metr, weights) for name, metr in metrics_dict.items()}
    cfg_ranking = sorted(cfg_scores.items(), key=lambda x: x[1], reverse=True)
    sensitivity_results[cfg_name] = {
        "scores": cfg_scores,
        "ranking": [n for n, _ in cfg_ranking],
        "top1": cfg_ranking[0][0],
        "top1_score": cfg_ranking[0][1],
    }

# ── Display Sensitivity Table ───────────────────────────────
models = list(all_results.keys())
W = 12

print("\n" + "="*90)
print("[RESULTS] WEIGHT SENSITIVITY ANALYSIS - COMPOSITE SCORES")
print("="*90)

header = f"  {'Configuration':<45}"
for m in models:
    header += f"{m[:W]:>{W}}"
header += f"  {'Top-1':<35}"
print(header)
print("  " + "-"*45 + "-"*W*len(models) + "  " + "-"*35)

cfg_ref = "Nominal        (0.30/0.25/0.20/0.15/0.10)"
top1_ref = sensitivity_results[cfg_ref]["top1"]

for cfg_name, res in sensitivity_results.items():
    line = f"  {cfg_name:<45}"
    top1 = res["top1"]
    stable = "[OK]" if top1 == top1_ref else "[!]"
    for n in models:
        score = res["scores"][n]
        flag = "*" if n == top1 else " "
        line += f"{flag+f'{score:.4f}':>{W}}"
    line += f"  {stable} {top1[:32]}"
    print(line)

print("  " + "-"*45 + "-"*W*len(models) + "  " + "-"*35)
print("  * = best score in configuration | [OK] = same ranking as nominal weights")

# ── Ranking Table ────────────────────────────────────────────
print("\n" + "="*90)
print("[RESULTS] RANKING TABLE BY CONFIGURATION")
print("="*90)

header2 = f"  {'Configuration':<45}"
for rank in range(1, len(models)+1):
    header2 += f"  {'Rank '+str(rank):<30}"
print(header2)
print("  " + "-"*45 + ("  " + "-"*30)*len(models))

for cfg_name, res in sensitivity_results.items():
    stable = "[OK]" if res["top1"] == top1_ref else "[!]"
    line = f"  {cfg_name:<45}"
    for rank, n in enumerate(res["ranking"]):
        score = res["scores"][n]
        line += f"  {n[:26]:<26} ({score:.4f})"
    line += f"  {stable}"
    print(line)

# ── Stability Analysis ──────────────────────────────────────
nb_configs = len(WEIGHT_CONFIGS)
nb_stable = sum(1 for r in sensitivity_results.values() if r["top1"] == top1_ref)
stability_rate = nb_stable / nb_configs * 100

top1_scores = [r["scores"][top1_ref] for r in sensitivity_results.values()]
max_gap = max(top1_scores) - min(top1_scores)

margins = []
for res in sensitivity_results.values():
    cl = res["ranking"]
    if len(cl) >= 2:
        margins.append(res["scores"][cl[0]] - res["scores"][cl[1]])
avg_margin = float(np.mean(margins)) if margins else 0.0
min_margin = float(np.min(margins)) if margins else 0.0

print("\n" + "="*70)
print("[REPORT] STABILITY REPORT (for paper)")
print("="*70)
print(f"  Reference model (nominal weights): {top1_ref}")
print(f"  Configurations tested            : {nb_configs}")
print(f"  Stable rankings (identical Top-1): {nb_stable}/{nb_configs} ({stability_rate:.0f}%)")
print(f"  Max score gap (Top-1)            : {max_gap:.4f}")
print(f"  Average margin Top1 vs Top2      : {avg_margin:.4f}")
print(f"  Minimum margin Top1 vs Top2      : {min_margin:.4f}")
print("="*70)

# ── Verdict ─────────────────────────────────────────────────
print("\n[VERDICT] FOR PAPER:")
if stability_rate == 100.0:
    print(f"  [OK] TOTAL INVARIANCE - ranking stable across all {nb_configs} configurations.")
    print(f"     -> Suggested paper sentence:")
    print(f"     'The ranking of {top1_ref[:40]}... is invariant")
    print(f"      across all {nb_configs} tested weight configurations (sensitivity range: {max_gap:.4f}),")
    print(f"      confirming that the composite score selection is not")
    print(f"      an artifact of the specific weighting scheme.'")
elif stability_rate >= 83.0:
    nb_unstable = nb_configs - nb_stable
    print(f"  [OK] QUASI-TOTAL INVARIANCE - {nb_stable}/{nb_configs} configurations stable.")
    print(f"     -> Suggested paper sentence:")
    print(f"     'The top-ranked model is consistent across {nb_stable} of")
    print(f"      {nb_configs} weight configurations; the {nb_unstable} exception(s)")
    print(f"      correspond to extreme weighting scenarios that diverge")
    print(f"      from the task-specific rationale detailed in the paper.'")
else:
    print(f"  [!] INSTABILITY DETECTED - only {nb_stable}/{nb_configs} configurations stable.")

# ── Average Contribution by Dimension ───────────────────────
print("\n[INFO] AVERAGE CONTRIBUTION BY DIMENSION (nominal weights):")
nom_weights = WEIGHT_CONFIGS[cfg_ref]
dims = ["bert_f1", "meteor", "rouge_l", "syn_val", "op_prec"]
dim_names = {
    "bert_f1": "BERT-F1   (w=0.30)",
    "meteor" : "METEOR    (w=0.25)",
    "rouge_l": "ROUGE-L   (w=0.20)",
    "syn_val": "Syn. Val. (w=0.15)",
    "op_prec": "Op. Prec. (w=0.10)",
}
for dim in dims:
    avg_contrib = np.mean([nom_weights[dim] * metr[dim] for metr in metrics_dict.values()])
    avg_val = np.mean([metr[dim] for metr in metrics_dict.values()])
    bar = "#" * int(avg_contrib * 100)
    print(f"  {dim_names[dim]:<22} : avg_val={avg_val:.4f} -> contrib={avg_contrib:.4f}  {bar}")

# ── Save Sensitivity Report ─────────────────────────────────
sensitivity_report = {
    "top1_reference": top1_ref,
    "nb_configs_tested": nb_configs,
    "nb_stable": nb_stable,
    "stability_rate_pct": stability_rate,
    "max_score_gap": max_gap,
    "avg_margin_top1_top2": avg_margin,
    "min_margin_top1_top2": min_margin,
    "configs": {
        cfg: {
            "weights": WEIGHT_CONFIGS[cfg],
            "top1": res["top1"],
            "scores": res["scores"],
            "ranking": res["ranking"],
        }
        for cfg, res in sensitivity_results.items()
    }
}
path_sens = f"{BASE_DIR}/results/sensitivity_analysis_weights.json"
with open(path_sens, "w", encoding="utf-8") as f:
    json.dump(sensitivity_report, f, indent=2, ensure_ascii=False)
print(f"\n[INFO] Sensitivity report saved to: {path_sens}")

This cell calculates the composite score for each configuration based on predefined weights and ranks them to automatically select the best evolution model for the ArchPipeline framework.

Composite Score Weights:

BERT-F1 (0.30) — Semantic quality
METEOR (0.25) — Lexical matching
ROUGE-L (0.20) — Sequential structure
Syntactic Validity (0.15) — ADL compliance
Global Operational Precision (0.10) — Correct transformation

In [ ]:
# ============================================================
# AUTOMATIC SELECTION OF THE BEST EVOLUTION MODEL
# ============================================================
import json
import os

def calculate_composite_score(res: dict) -> float:
    """Calculates the weighted composite score for a model."""
    val = res.get("syntactic_validity", {}).get("global_validity", 0.0)
    prec = res.get("operational_precision", {}).get("GLOBAL", {}).get("precision", 0.0)
    return round(
        0.30 * res.get("BERT-F1", 0.0) +
        0.25 * res.get("METEOR", 0.0) +
        0.20 * res.get("ROUGE-L", 0.0) +
        0.15 * val +
        0.10 * prec
    , 4)

# Ensure all_results is available
if 'all_results' not in globals() or not all_results:
    raise RuntimeError("[ERROR] Variable 'all_results' not found. Please run previous cells first.")

scores = {name: calculate_composite_score(res) for name, res in all_results.items()}
ranking = sorted(scores.items(), key=lambda x: x[1], reverse=True)

print("\n" + "="*72)
print("[RESULTS] FINAL RANKING - ADL EVOLUTION MODEL (ArchPipeline)")
print("="*72)

for i, (name, sc) in enumerate(ranking):
    rank_label = f"[{i+1}]"
    res = all_results[name]
    bf1 = res.get("BERT-F1", 0.0)
    meteor_v = res.get("METEOR", 0.0)
    rL = res.get("ROUGE-L", 0.0)
    val = res.get("syntactic_validity", {}).get("global_validity", 0.0)
    prec = res.get("operational_precision", {}).get("GLOBAL", {}).get("precision", 0.0)
    
    print(f"  {rank_label} {name:<42} score={sc:.4f}")
    print(f"      BERT-F1={bf1:.4f} | METEOR={meteor_v:.4f} | ROUGE-L={rL:.4f}")
    print(f"      Syntactic Validity={val:.4f} | Op. Precision={prec:.4f}")
    print()

print("="*72)
best_model = ranking[0][0]
print(f"\n[SELECTED MODEL] ArchPipeline Evolution Backbone: {best_model}")
print(f"   Path  : {CONFIGS[best_model]}")
print(f"   Score : {scores[best_model]:.4f}")
print(f"\n   This model will be used as the final generative backbone for ArchPipeline.")

# Save the selection report
selection_report = {
    "best_model": best_model,
    "path": CONFIGS[best_model],
    "composite_score": scores[best_model],
    "ranking": [{"model": n, "score": s} for n, s in ranking],
    "all_results": all_results,
}
path_report = f"{BASE_DIR}/results/evolution_selection_final.json"
with open(path_report, "w", encoding="utf-8") as f:
    json.dump(selection_report, f, indent=2, ensure_ascii=False)
print(f"\n[INFO] Selection report saved to: {path_report}")

Paired Comparison on 20 Instances (Reviewer 2 Response)
To address the sample size asymmetry highlighted by Reviewer 2, this cell reproduces the exact stratified 20-instance subset (seed=42) used for the LLM baselines. We re-evaluate ArchPipeline (Model 1) on this specific subset to ensure a fair, sample-matched comparison.

All related data (sampled IDs, scores, comparison tables) are saved to the results/paired_comparison_20/ directory for full transparency.

In [ ]:
# ============================================================
# REVIEWER 2 RESPONSE — Fair comparison on the same 20 instances
# Reproduces EXACTLY the same stratified sampling (seed=42)
# used for LLM baselines, then evaluates ArchPipeline on it.
# ============================================================
import random
import os
import json
from collections import defaultdict
from datetime import datetime

# --- 1. Reload the full test set (same file as baselines) ---
_, _, test_data = load_evolution_datasets()

# Add stable position ID for traceability
for i, ex in enumerate(test_data):
    ex["_id"] = i

# --- 2. Reproduce EXACTLY the same stratified sampling ---
random.seed(42)

ops_dict_test = defaultdict(list)
for ex in test_data:
    op = ex.get("operation", ex["input"].split("[OP]")[1].split("[")[0].strip())
    ops_dict_test[op].append(ex)

nb_ops = len(ops_dict_test)
n_total = 20
n_per_op = n_total // nb_ops
n_remainder = n_total % nb_ops

ops_sorted = sorted(ops_dict_test.items(), key=lambda x: -len(x[1]))

test_subset_20 = []
for i, (op, examples) in enumerate(ops_sorted):
    quota = n_per_op + (1 if i < n_remainder else 0)
    quota = min(quota, len(examples))
    test_subset_20.extend(random.sample(examples, quota))

random.shuffle(test_subset_20)

print(f"[INFO] Subset reproduced: {len(test_subset_20)} instances")
print("Selected IDs :", sorted(ex["_id"] for ex in test_subset_20))
print()

# --- 3. Evaluate ArchPipeline (Model 1, CodeT5-base StratA) on THESE 20 instances ---
results_20 = evaluate_evolution_model(
    model_ct5b_A, tokenizer_ct5b, test_subset_20,
    model_name="CodeT5base_StratA_PAIRED20",
    output_dir=OUT_CT5B_A
)

# --- 4. Side-by-side comparison: 120 (full) vs 20 (paired with baselines) ---
print("\n" + "="*70)
print("[RESULTS] COMPARISON — ArchPipeline on 120 (full) vs 20 (paired)")
print("="*70)
print(f"{'Metric':<20}{'n=120':>12}{'n=20 (paired)':>18}{'Delta':>10}")
print("-"*70)

for key, label in [("EM","EM"), ("BLEU","BLEU"), ("ROUGE-L","ROUGE-L"),
                    ("METEOR","METEOR"), ("BERT-F1","BERT-F1"),
                    ("composite_score","Scomp")]:
    v120 = results_ct5b_A.get(key, float('nan'))
    v20  = results_20.get(key, float('nan'))
    print(f"{label:<20}{v120:>12.4f}{v20:>18.4f}{v20-v120:>10.4f}")

# --- 5. Full save for GitHub publication (reviewer transparency) ---
save_dir = f"{OUT_CT5B_A}/paired_comparison_20"
os.makedirs(save_dir, exist_ok=True)

# 5a. IDs + metadata of the 20 selected instances
ids_detail = [
    {"id": ex["_id"], "operation": ex["operation"],
     "adl_type": ex["adl_type"], "difficulte": ex["difficulte"],
     "source": ex.get("source", ""), "distance_train": ex.get("distance_train", None)}
    for ex in test_subset_20
]
with open(f"{save_dir}/sampled_20_instance_ids.json", "w", encoding="utf-8") as f:
    json.dump(ids_detail, f, indent=2, ensure_ascii=False)

# 5b. ArchPipeline full scores on the 20 instances
with open(f"{save_dir}/archpipeline_scores_n20_paired.json", "w", encoding="utf-8") as f:
    json.dump(results_20, f, indent=2, ensure_ascii=False)

# 5c. ArchPipeline full scores (n=120) for reference/comparison
with open(f"{save_dir}/archpipeline_scores_n120_full.json", "w", encoding="utf-8") as f:
    json.dump(results_ct5b_A, f, indent=2, ensure_ascii=False)

# 5d. Synthetic comparison table 120 vs 20, JSON format
comparison_table = {
    "metadata": {
        "generated_at": datetime.now().isoformat(),
        "seed": 42,
        "n_full": 120,
        "n_paired": 20,
        "sampling_strategy": "stratified by operation type (3-4 per operation, seed=42)"
    },
    "metrics": {}
}
for key, label in [("EM","EM"), ("BLEU","BLEU"), ("ROUGE-L","ROUGE-L"),
                    ("METEOR","METEOR"), ("BERT-F1","BERT-F1"),
                    ("composite_score","Scomp")]:
    v120 = results_ct5b_A.get(key, None)
    v20  = results_20.get(key, None)
    comparison_table["metrics"][label] = {
        "n120": v120, "n20_paired": v20,
        "delta": (v20 - v120) if (v120 is not None and v20 is not None) else None
    }

with open(f"{save_dir}/comparison_n120_vs_n20.json", "w", encoding="utf-8") as f:
    json.dump(comparison_table, f, indent=2, ensure_ascii=False)

# 5e. Human-readable text version (for README / supplementary material)
with open(f"{save_dir}/comparison_n120_vs_n20.txt", "w", encoding="utf-8") as f:
    f.write("ArchPipeline (CodeT5-base StratA) - Paired Comparison\n")
    f.write("="*70 + "\n")
    f.write(f"Full test set (n=120) vs. stratified subset matching LLM baselines (n=20, seed=42)\n\n")
    f.write(f"{'Metric':<20}{'n=120':>12}{'n=20 (paired)':>18}{'Delta':>10}\n")
    f.write("-"*70 + "\n")
    for label, vals in comparison_table["metrics"].items():
        f.write(f"{label:<20}{vals['n120']:>12.4f}{vals['n20_paired']:>18.4f}{vals['delta']:>10.4f}\n")
    f.write("\nSampled instance IDs (position index in evo_test_v2.jsonl):\n")
    f.write(str(sorted(ex["_id"] for ex in test_subset_20)) + "\n")

print(f"\n[INFO] All files saved in: {save_dir}")
print("   - sampled_20_instance_ids.json")
print("   - archpipeline_scores_n20_paired.json")
print("   - archpipeline_scores_n120_full.json")
print("   - comparison_n120_vs_n20.json")
print("   - comparison_n120_vs_n20.txt")
